<a href="https://colab.research.google.com/github/HasanAyaz058/flyrank-ml-internship/blob/main/w07_action_playbook_completed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

This notebook turns the validated Week-5 content opportunity model into a practical, human-reviewed action queue. It is intentionally non-production and uses public-safe, evidence-first language.

## 1. Ranked actions + reason codes

### Purpose
This playbook turns the Week-5/Week-6 validated content-opportunity model output into a **ranked human-review queue**.

The selected model is Logistic Regression. Its score is treated as a **directional prioritization signal**, not as a guaranteed prediction or a guarantee of refresh ROI.

### Action logic
Pages with higher model probability are reviewed first. Reason codes add human-readable context using only information available at the decision moment:

- `HIGH_DECLINE_SIGNAL` — higher measured model probability of the selected future-decline label.
- `STALE` — content is at least 180 days old at the snapshot.
- `POSITION_SLIP` — average position worsened by more than 2 positions versus the previous 30-day window.
- `VISIBLE_VOLUME` — current 30-day impressions meet the minimum volume threshold.
- `LOW_SEARCH_VOLUME` — search-volume context is low; review the signal carefully rather than assuming a refresh is valuable.
- `THIN_CONTENT` — word count is relatively low; this is a review cue, not proof that the page is poor.

### Ranked queue
The code below recreates the Week-5/Week-6 model on the March snapshot, ranks pages by model probability, attaches reason codes, and prints the first 20 anonymized review candidates. No client names, URLs, queries, titles, or domains are exported.


In [ ]:
import os, json, numpy as np, pandas as pd, duckdb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
import matplotlib.pyplot as plt

SEED = 42
MIN_IMPRESSIONS = 100
MIN_DAYS = 20
DECLINE_THRESHOLD = 0.80

# Public-repo safe access: token comes from Colab Secrets, never from notebook text.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError(
        "Add the Hugging Face READ token as a Colab Secret named HF_TOKEN "
        "before running this notebook."
    )

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
MONTHS = {
    m: f"read_parquet('{REL}/fact_content_daily_performance/month={m}/*.parquet')"
    for m in ["2026-01", "2026-02", "2026-03", "2026-04"]
}

parts = []
for src in MONTHS.values():
    parts.append(f"""
    SELECT report_date, client_hash_id, content_hash_id, gsc_data_available,
           gsc_impressions, gsc_clicks,
           CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END AS gsc_avg_position
    FROM {src}
    WHERE gsc_data_available IS TRUE
    """)

monthly = con.sql(f"""
WITH d AS ({' UNION ALL '.join(parts)})
SELECT DATE_TRUNC('month', report_date) AS month,
       client_hash_id, content_hash_id,
       SUM(gsc_impressions) AS impressions,
       SUM(gsc_clicks) AS clicks,
       AVG(gsc_avg_position) AS avg_position,
       COUNT(DISTINCT report_date) AS gsc_days
FROM d
GROUP BY 1,2,3
""").df()

monthly["month"] = pd.to_datetime(monthly["month"]).dt.strftime("%Y-%m")

content = con.sql(f"""
SELECT client_hash_id, content_hash_id, content_updated_date,
       search_volume, word_count, content_type, is_published, is_deleted
FROM {CONTENT}
""").df()

def make_snapshot(s):
    p = (pd.Period(s, "M") - 1).strftime("%Y-%m")
    n = (pd.Period(s, "M") + 1).strftime("%Y-%m")
    cur = monthly[monthly.month == s].rename(columns={
        "impressions":"impressions_current30",
        "clicks":"clicks_current30",
        "avg_position":"position_current30",
        "gsc_days":"gsc_days_current30"
    })
    prev = monthly[monthly.month == p].rename(columns={
        "impressions":"impressions_prev30",
        "clicks":"clicks_prev30",
        "avg_position":"position_prev30",
        "gsc_days":"gsc_days_prev30"
    })
    nxt = monthly[monthly.month == n].rename(columns={
        "impressions":"impressions_next30",
        "clicks":"clicks_next30",
        "avg_position":"position_next30",
        "gsc_days":"gsc_days_next30"
    })
    x = cur.merge(prev, on=["client_hash_id","content_hash_id"]).merge(
        nxt[["client_hash_id","content_hash_id","impressions_next30",
             "clicks_next30","position_next30","gsc_days_next30"]],
        on=["client_hash_id","content_hash_id"]
    ).merge(content, on=["client_hash_id","content_hash_id"], how="left")

    end = pd.Timestamp(s + "-01") + pd.offsets.MonthEnd(0)
    x["staleness_days"] = (end - pd.to_datetime(x.content_updated_date)).dt.days
    x["position_slip"] = x.position_current30 - x.position_prev30
    x["ctr_current30"] = x.clicks_current30 / x.impressions_current30.replace(0, np.nan)
    x["ctr_prev30"] = x.clicks_prev30 / x.impressions_prev30.replace(0, np.nan)

    x = x[
        (x.is_published == True) & (x.is_deleted == False) &
        (x.gsc_days_prev30 >= MIN_DAYS) &
        (x.gsc_days_current30 >= MIN_DAYS) &
        (x.gsc_days_next30 >= MIN_DAYS) &
        (x.impressions_prev30 >= MIN_IMPRESSIONS) &
        (x.impressions_current30 >= MIN_IMPRESSIONS)
    ].copy()

    x["is_declining"] = (
        x.impressions_next30 < DECLINE_THRESHOLD * x.impressions_current30
    ).astype(int)
    x["snapshot_month"] = s
    return x

train_df = make_snapshot("2026-02")
test_df = make_snapshot("2026-03")

feature_cols = [
    "impressions_prev30","impressions_current30",
    "clicks_prev30","clicks_current30",
    "position_prev30","position_current30","position_slip",
    "ctr_prev30","ctr_current30","staleness_days",
    "search_volume","word_count"
]

for frame in [train_df, test_df]:
    for c in feature_cols:
        frame[c] = pd.to_numeric(frame[c], errors="coerce")
    frame[feature_cols] = frame[feature_cols].replace([np.inf, -np.inf], np.nan)

def make_model():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=SEED
        ))
    ])

# Honest model selection: grouped by client on the training snapshot.
gss = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=SEED)
tr_idx, va_idx = next(gss.split(
    train_df, train_df.is_declining, groups=train_df.client_hash_id
))
tr = train_df.iloc[tr_idx]
va = train_df.iloc[va_idx]

model = make_model().fit(tr[feature_cols], tr.is_declining)
va_prob = model.predict_proba(va[feature_cols])[:, 1]

print("Training rows:", len(tr))
print("Grouped validation rows:", len(va))
print("Grouped validation average precision:",
      round(average_precision_score(va.is_declining, va_prob), 3))

# The March snapshot is the held-out, time-aware outcome window used by Week 6.
test_prob = model.predict_proba(test_df[feature_cols])[:, 1]
print("March test rows:", len(test_df))
print("March test average precision:",
      round(average_precision_score(test_df.is_declining, test_prob), 3))
print("March test ROC AUC:",
      round(roc_auc_score(test_df.is_declining, test_prob), 3))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# Build an anonymized ranked queue from the held-out March snapshot.
queue = test_df[[
    "impressions_current30", "position_current30", "position_slip",
    "staleness_days", "search_volume", "word_count"
]].copy()

queue["model_probability"] = test_prob

def reason_codes(row):
    reasons = []
    if row["model_probability"] >= queue["model_probability"].quantile(0.75):
        reasons.append("HIGH_DECLINE_SIGNAL")
    if row["staleness_days"] >= 180:
        reasons.append("STALE")
    if row["position_slip"] > 2:
        reasons.append("POSITION_SLIP")
    if row["impressions_current30"] >= MIN_IMPRESSIONS:
        reasons.append("VISIBLE_VOLUME")
    if pd.notna(row["search_volume"]) and row["search_volume"] < 10:
        reasons.append("LOW_SEARCH_VOLUME")
    if pd.notna(row["word_count"]) and row["word_count"] < 500:
        reasons.append("THIN_CONTENT")
    return ";".join(reasons) if reasons else "NO_STRONG_SUPPORTING_CUE"

queue["reason_codes"] = queue.apply(reason_codes, axis=1)

def suggested_action(row):
    stale = row["staleness_days"] >= 180
    slip = row["position_slip"] > 2
    visible = row["impressions_current30"] >= MIN_IMPRESSIONS
    low_volume = pd.notna(row["search_volume"]) and row["search_volume"] < 10
    thin = pd.notna(row["word_count"]) and row["word_count"] < 500

    if low_volume:
        return "INVESTIGATE_FIRST"
    if stale and slip and visible:
        return "REVIEW_REFRESH"
    if slip and visible:
        return "INVESTIGATE_CONTEXT"
    if stale:
        return "MONITOR_OR_REVIEW"
    if thin:
        return "CHECK_COMPLETENESS"
    return "MONITOR"

queue["suggested_action"] = queue.apply(suggested_action, axis=1)

queue = queue.sort_values(
    ["model_probability", "impressions_current30", "staleness_days"],
    ascending=[False, False, False],
    kind="stable"
).reset_index(drop=True)

queue.insert(0, "review_rank", np.arange(1, len(queue) + 1))

# Keep only public-safe, useful review fields.
queue = queue[[
    "review_rank", "model_probability", "suggested_action", "reason_codes",
    "impressions_current30", "position_current30", "position_slip",
    "staleness_days", "search_volume", "word_count"
]]

print(queue.head(20).to_string(index=False))


## 2. Intended use and limits

### Intended use
The queue is a **decision-support tool for human review**. It helps a reviewer decide which pages to inspect first when looking for possible content-refresh opportunities.

The model's target is the selected Week-5 label: whether next-30-day impressions fall below 80% of current-30-day impressions. The model uses only pre-decision features such as recent impressions, clicks, CTR, position, staleness, search volume, and word count.

### Limits
- The score is directional; it does not prove that a page will decline.
- A high score does not prove that refreshing the page will improve traffic or SEO performance.
- The evaluated March snapshot is one time-aware test period; it does not establish universal performance across all future periods or clients.
- Observational data can reflect seasonality, search-system changes, external demand, consolidation, or measurement noise.
- Reason codes are review cues, not causal explanations.
- The queue is intentionally non-production and should not directly trigger publishing or content changes.


## 3. Human review + the no-go list

### Human review before action
For each high-priority candidate, a reviewer should check:

1. Is the page still strategically relevant?
2. Is the observed decline/visibility context real enough to investigate, rather than low-volume noise?
3. Is the content stale in a meaningful way, or is its age appropriate for the topic?
4. Does the search/position context suggest an actual content opportunity?
5. Would a proposed change preserve useful information and search intent?
6. Are there business, legal, editorial, or product constraints that the model cannot see?

### Archetype → action mapping
| Review archetype | Typical evidence | Suggested human action |
|---|---|---|
| Stale + visible + slipping | Older content, sufficient impressions, position worsened | Review for refresh/update |
| Visible + slipping, not especially stale | Measurable movement but limited age signal | Investigate intent, SERP/context, and content fit |
| Stale + stable | Old content without a strong decline signal | Monitor; refresh only if content is materially outdated |
| Low-volume / uncertain | Weak demand or noisy measurements | Investigate before investing effort |
| Thin-content review cue | Low word count plus other opportunity signals | Check completeness and intent match |
| No strong signal | Low model priority and weak supporting cues | Monitor rather than force an action |

### What should NOT be automated
The model must **not** automatically:
- publish or rewrite content;
- delete, merge, or redirect pages;
- change titles/meta descriptions;
- declare that a refresh will improve rankings or traffic;
- override editorial/business judgment;
- expose client, URL, query, title, domain, or other private examples.

The queue recommends where to look first. A person decides what to do.


## 4. Monitoring / retrain triggers

The model should be treated as stale when its measured behavior or its input environment changes.

### Light monitoring
Review the model when:
- average precision on a later time-aware evaluation falls materially below the prior observed level;
- Precision@20 or Precision@50 falls enough that the review queue is no longer useful;
- the base rate of the future-decline label changes substantially;
- the distribution of key features such as impressions, staleness, or position changes materially;
- important input fields become missing or their measurement definition changes;
- the relationship between model scores and later outcomes changes.

### Retrain / re-audit trigger
Retrain only after a fresh leakage check and a new time-aware or client-grouped validation. A new model should not be promoted simply because it scores higher on one convenient split.

The practical trigger is therefore: **new data + evidence of drift or degraded usefulness → re-audit → revalidate → compare with the existing model → decide whether to replace it.**


## 5. Exports for the paper

The notebook creates two reusable artifacts:

1. `work/outputs/w07_action_queue.csv` — the ranked, anonymized review queue. This is generated locally and should remain out of Git according to the assignment's data-leak guard.
2. `work/outputs/w07_action_playbook_metrics.json` — compact, reproducible metrics/metadata for tracing the paper's claims.
3. `work/figures/w07_action_counts.png` — a simple figure showing the number of candidates receiving each suggested action.

The queue contains no client names, URLs, raw queries, titles, or domains.


In [ ]:
# Export artifacts for the paper.
out_dir = os.path.join(os.getcwd(), "work", "outputs")
fig_dir = os.path.join(os.getcwd(), "work", "figures")
os.makedirs(out_dir, exist_ok=True)
os.makedirs(fig_dir, exist_ok=True)

queue_path = os.path.join(out_dir, "w07_action_queue.csv")
metrics_path = os.path.join(out_dir, "w07_action_playbook_metrics.json")
figure_path = os.path.join(fig_dir, "w07_action_counts.png")

queue.to_csv(queue_path, index=False)

metrics = {
    "seed": SEED,
    "snapshot": "2026-03 -> April outcome",
    "selected_model": "Logistic Regression",
    "validation": "GroupShuffleSplit by client_hash_id on 2026-02 training snapshot",
    "target": "next30 impressions < 0.80 * current30 impressions",
    "test_rows": int(len(test_df)),
    "test_base_rate": float(test_df.is_declining.mean()),
    "test_average_precision": float(average_precision_score(test_df.is_declining, test_prob)),
    "test_roc_auc": float(roc_auc_score(test_df.is_declining, test_prob)),
    "queue_rows": int(len(queue)),
    "top20_average_model_probability": float(queue.head(20).model_probability.mean()),
    "public_safe_export": True
}
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

action_counts = queue["suggested_action"].value_counts().sort_values(ascending=False)
plt.figure(figsize=(8, 4.5))
action_counts.plot(kind="bar")
plt.title("Week-7 Suggested Action Counts")
plt.xlabel("Suggested action")
plt.ylabel("Number of review candidates")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(figure_path, dpi=160)
plt.close()

print("Queue:", queue_path)
print("Metrics:", metrics_path)
print("Figure:", figure_path)


In [ ]:
# Final automated checks.
future_or_label = {
    "is_declining", "impressions_next30", "clicks_next30",
    "position_next30", "gsc_days_next30", "trend_pct",
    "trend_direction", "is_declining_label"
}
product_outputs = {"health_score", "priority_score", "action_type", "decision_flag"}

assert future_or_label.isdisjoint(feature_cols)
assert product_outputs.isdisjoint(feature_cols)
assert "client_hash_id" not in feature_cols
assert set(test_df.snapshot_month) == {"2026-03"}

assert os.path.exists(queue_path)
assert os.path.exists(metrics_path)
assert os.path.exists(figure_path)

unsafe_columns = {
    "client_hash_id", "client_id", "url", "query", "query_hash",
    "title", "domain", "keyword", "category"
}
assert unsafe_columns.isdisjoint(queue.columns)

print("SELF-CHECK: PASS")
print("- Pre-decision features only: PASS")
print("- Future/label fields excluded from features: PASS")
print("- Client IDs used only for grouping: PASS")
print("- Public-unsafe identifiers excluded from queue: PASS")
print("- Queue exported: PASS")
print("- Metrics receipt exported: PASS")
print("- Reusable figure exported: PASS")


## Self-check

- [x] Ranked actions are tied to the validated Week-5 model.
- [x] Reason codes explain the ranking using pre-decision signals.
- [x] Intended use is decision-support, not autonomous production action.
- [x] Limits are stated using observed/measured/directional language.
- [x] Human review rules are explicit.
- [x] No-go cases say what must not be automated.
- [x] Archetypes are mapped to practical review actions.
- [x] Monitoring and retrain triggers are defined.
- [x] Queue and reusable paper artifacts are exported.
- [x] No client names, URLs, raw queries, titles, or domains are exported.
- [ ] Run Runtime → Run all in Colab with the repository data access available.
- [ ] Confirm the generated files are present and commit the notebook/allowed receipts, not the queue CSV.
